<a href="https://colab.research.google.com/github/rasika0405/notebooks-01_preprocessing_eda.ipynb/blob/main/Welcome_to_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Welcome to Colab!

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

uploaded = files.upload()
filename = next(iter(uploaded))

df = pd.read_csv(filename, encoding="latin1", low_memory=False)
df.columns = df.columns.str.strip()

if "Label" not in df.columns:
    raise ValueError("The selected CSV needs a 'Label' column.")

print("Records and columns:", df.shape)

print("\nAvailable columns:")
print(df.columns.tolist())

print("\nLabel counts:")
print(df["Label"].astype("string").str.strip().value_counts(dropna=False))

print("\nMissing values:", df.isna().sum().sum())
numeric = df.select_dtypes(include="number")
print("Infinite values:", np.isinf(numeric).sum().sum())

display(df.head())

In [ ]:
if "Label" not in df.columns:
    raise ValueError("The CSV needs a 'Label' column.")

print("Records:", df.shape[0])
print("Columns:", df.shape[1])

print("\nNormal and attack counts:")
print(df["Label"].astype("string").str.strip().value_counts(dropna=False))

print("\nMissing values:", df.isna().sum().sum())

numeric = df.select_dtypes(include="number")
print("Infinite values:", np.isinf(numeric).sum().sum())

print("Exact duplicate rows:", df.duplicated().sum())

print("\nColumn names:")
print(df.columns.tolist())

display(df.head())

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

# 1. Separate the known answers from the measurements.
labels = df["Label"].astype("string").str.strip()

if not labels.isin(["BENIGN", "PortScan"]).all():
    raise ValueError("Missing or unexpected labels need investigation.")

features = df.drop(columns="Label").apply(
    pd.to_numeric, errors="coerce"
)

# 2. Treat infinite values as missing.
features = features.replace([np.inf, -np.inf], np.nan)

# 3. Remove the extra header column only if it is identical.
if (
    "Fwd Header Length.1" in features.columns
    and "Fwd Header Length" in features.columns
    and features["Fwd Header Length.1"].equals(
        features["Fwd Header Length"]
    )
):
    features = features.drop(columns="Fwd Header Length.1")
    print("Removed an identical extra header column.")

# 4. Keep one copy of each identical measurement-and-label row.
prepared = pd.concat(
    [features, labels.rename("Label")], axis=1
)
before = len(prepared)
prepared = prepared.drop_duplicates().reset_index(drop=True)

print("Repeated rows removed:", before - len(prepared))
print("Records remaining:", len(prepared))

# X = measurements; y = answer: 0 normal, 1 PortScan.
X = prepared.drop(columns="Label")
y = prepared["Label"].map({"BENIGN": 0, "PortScan": 1}).astype(int)

# 5. Give identical measurement rows the same group ID.
# This also keeps identical inputs with different labels together.
groups = pd.util.hash_pandas_object(X, index=False).to_numpy()

first_split = GroupShuffleSplit(
    n_splits=1, test_size=0.30, random_state=42
)
train_idx, remaining_idx = next(
    first_split.split(X, y, groups=groups)
)

second_split = GroupShuffleSplit(
    n_splits=1, test_size=0.50, random_state=42
)
val_relative, test_relative = next(
    second_split.split(
        X.iloc[remaining_idx],
        groups=groups[remaining_idx]
    )
)

val_idx = remaining_idx[val_relative]
test_idx = remaining_idx[test_relative]
